# 04 - Text Splitter / Chunking


## What this notebook is for

This notebook is written as **notes + code**. Read the Markdown cells first, then run the code cells.

The goal is to understand the concept, see the architecture, implement a small example, and then evaluate/improve it.

## Learning objectives

- Understand why chunking affects RAG quality.
- Compare fixed, overlap, sentence, recursive, token, and semantic chunking.
- Use LangChain splitters.
- Measure chunk size and inspect bad splits.

## Core AI concept

Chunking creates retrieval units. A chunk that is too small may lose context; a chunk that is too large may contain noise and reduce retrieval precision. Chunking is therefore a retrieval design problem, not merely a preprocessing trick.

## Architecture / mental model

```text
Raw document
    ↓
Chunking strategy
    ↓
Meaningful chunks + metadata
    ↓
Embedding
    ↓
Retriever
```

## Connection to the supplied learning material

Your `chunking.ipynb` already demonstrates fixed, overlap, sentence, recursive, token, and semantic strategies. This notebook keeps that progression and adds inspection/evaluation.

### Source foundation
This notebook is an expanded learning version of the uploaded notebooks. The source material for this track includes:
- `chunking.ipynb`
- `embeddings.ipynb`
- `faiss.ipynb`
- `chroma_db.ipynb`
- `searching.ipynb`
- `rag.ipynb`

The original examples and terminology are retained where useful, then organized into a cleaner learning sequence.

## Setup

The code below uses environment variables for secrets and model names so the notebook is easier to maintain when providers or model versions change.

In [ ]:
from pathlib import Path
import os
from dotenv import load_dotenv

load_dotenv()

OPENAI_MODEL = os.getenv("OPENAI_MODEL", "gpt-5.6-luna")
OPENAI_EMBED_MODEL = os.getenv("OPENAI_EMBED_MODEL", "text-embedding-3-small")

print("Model:", OPENAI_MODEL)
print("Embedding model:", OPENAI_EMBED_MODEL)

### Install splitters

In [ ]:
%pip install -qU langchain-text-splitters

### Sample document

In [ ]:
text = """
Artificial intelligence is a broad field.
Machine learning is a subset of AI.
Deep learning uses neural networks with many layers.
RAG combines retrieval with generation.
Agents can use tools to take actions.
"""

### Recursive splitter

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=100,
    chunk_overlap=20,
    separators=["\n\n", "\n", ". ", " ", ""],
)

chunks = splitter.split_text(text)

for i, chunk in enumerate(chunks):
    print(i, len(chunk), repr(chunk))

### Markdown-aware splitting

In [ ]:
from langchain_text_splitters import MarkdownHeaderTextSplitter

markdown = """# AI
## RAG
RAG retrieves external knowledge.
## Agents
Agents can call tools.
"""

md_splitter = MarkdownHeaderTextSplitter(
    headers_to_split_on=[("#", "h1"), ("##", "h2")]
)

docs = md_splitter.split_text(markdown)

for doc in docs:
    print(doc.metadata, doc.page_content)

### Simple chunk inspection

In [ ]:
def inspect_chunks(chunks):
    sizes = [len(c) for c in chunks]
    return {
        "count": len(chunks),
        "min_chars": min(sizes),
        "max_chars": max(sizes),
        "avg_chars": sum(sizes)/len(sizes),
    }

print(inspect_chunks(chunks))

## Evaluation

Build a small gold set of questions and expected supporting chunks. Compare recall@k across chunk sizes/overlaps. Inspect whether important concepts are split across boundaries.

## Improvement ideas

Tune chunk size/overlap on evaluation data, preserve headings, keep table rows together, use code-aware splitting for code, and use semantic or document-aware splitting when fixed sizes fail.

## Practice exercises

1. Compare chunk sizes 100, 250, 500.
2. Add a sentence-based splitter.
3. Measure retrieval recall for each strategy.

## Final checklist

- [ ] I can explain the concept in simple words.
- [ ] I can describe where it fits in a GenAI system.
- [ ] I can run and modify the code.
- [ ] I know at least one way to evaluate it.
- [ ] I know what to change when it fails.

## Research / reference notes


- LangChain Python reference: https://reference.langchain.com/python/langchain/overview
- LangGraph Python reference: https://reference.langchain.com/python/langgraph/overview
- Ragas metrics: https://docs.ragas.io/en/latest/concepts/metrics/available_metrics/
- OpenAI function calling guide: https://developers.openai.com/api/docs/guides/function-calling
- OpenAI Responses API: https://developers.openai.com/api/reference/cli/resources/responses/methods/create

- LangChain RecursiveCharacterTextSplitter: https://reference.langchain.com/python/langchain-text-splitters/character/RecursiveCharacterTextSplitter